# AI Data Analyst: Evals + MCP Demo

This notebook demonstrates the new local eval runner and MCP helper layer added to the working copy of the project.

Project folder: `/Users/akshay/Desktop/ai-data-analyst-work`

What was added:

- `evals/local_runner.py` for offline SQL scorer evaluation
- `src/mcp_server/server.py` for project-specific MCP tools/resources
- `docs/MCP.md` with MCP usage notes
- unit tests for local evals and MCP helpers
- Poetry scripts: `ai-data-analyst-evals` and `ai-data-analyst-mcp`

Verification status:

- Python syntax compilation passed for the new files.
- `poetry check` passed with only existing deprecation warnings.
- `poetry.lock` was refreshed.
- Full dependency install and pytest execution were blocked because the disk has only a few hundred MB free.

In [ ]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path('/Users/akshay/Desktop/ai-data-analyst-work')
sys.path.insert(0, str(PROJECT_ROOT))

print(PROJECT_ROOT)
print('pyproject exists:', (PROJECT_ROOT / 'pyproject.toml').exists())
print('MCP server exists:', (PROJECT_ROOT / 'src/mcp_server/server.py').exists())
print('Local eval runner exists:', (PROJECT_ROOT / 'evals/local_runner.py').exists())

## 1. Confirm the command wiring

The project now exposes two Poetry commands:

- `poetry run ai-data-analyst-evals`
- `poetry run ai-data-analyst-mcp`

In [ ]:
pyproject = (PROJECT_ROOT / 'pyproject.toml').read_text()

for line in pyproject.splitlines():
    if 'mcp =' in line or 'ai-data-analyst-' in line or '[tool.poetry.scripts]' in line:
        print(line)

## 2. Run the local SQL eval baseline

This uses the existing SQL eval dataset and scores the expected SQL with the local validity/efficiency scorers. It does not call an LLM and does not need a database.

In [ ]:
try:
    from evals.local_runner import run_local_eval

    payload = run_local_eval('sql_generation')
    print(json.dumps(payload['summary'], indent=2))
except ModuleNotFoundError as exc:
    print('Missing dependency:', exc)
    print('After freeing disk space, run: poetry install')

## 3. Score a single SQL query

The MCP helper exposes a reusable `score_sql` function. It blocks obvious mutation queries and gives a combined score for read-only SQL quality.

In [ ]:
try:
    from src.mcp_server.server import score_sql

    safe_query = 'SELECT id, name FROM customers LIMIT 10'
    unsafe_query = 'DROP TABLE customers'

    print('Safe query score:')
    print(json.dumps(score_sql(safe_query, tables=['customers']), indent=2))
    print('\nUnsafe query score:')
    print(json.dumps(score_sql(unsafe_query), indent=2))
except ModuleNotFoundError as exc:
    print('Missing dependency:', exc)
    print('After freeing disk space, run: poetry install')

## 4. Run non-LLM analysis on rows

The MCP helper can run the existing `StatsToolkit` on caller-provided rows. This demonstrates analysis without hitting OpenAI or PostgreSQL.

In [ ]:
try:
    from src.mcp_server.server import analyze_rows

    rows = [
        {'category': 'Books', 'revenue': 1200, 'orders': 40},
        {'category': 'Electronics', 'revenue': 3400, 'orders': 55},
        {'category': 'Home', 'revenue': 2100, 'orders': 35},
    ]
    result = analyze_rows(rows, columns=['category', 'revenue', 'orders'])
    print(json.dumps(result, indent=2, default=str))
except ModuleNotFoundError as exc:
    print('Missing dependency:', exc)
    print('After freeing disk space, run: poetry install')

## 5. Inspect MCP project context

The MCP server exposes project metadata and an allow-listed resource reader. It does not expose arbitrary file access.

In [ ]:
try:
    from src.mcp_server.server import project_info, read_project_file

    print(json.dumps(project_info(), indent=2))
    print('\nREADME preview:')
    print(read_project_file('README.md')[:500])
except ModuleNotFoundError as exc:
    print('Missing dependency:', exc)
    print('After freeing disk space, run: poetry install')

## 6. Commands to run after freeing disk space

Run these from `/Users/akshay/Desktop/ai-data-analyst-work`:

```bash
poetry install
poetry run ai-data-analyst-evals
poetry run pytest tests/unit/test_local_evals.py tests/unit/test_mcp_server.py -v
poetry run ai-data-analyst-mcp
```

Current blocker: the disk was at 100% capacity during dependency installation, so the full environment could not be installed yet.